# Chest X-Ray Image Classification (Research Prototype)

A TensorFlow/Keras transfer-learning notebook for four-way chest X-ray image classification: **normal**, **other**, **pneumonia**, and **tuberculosis**. It combines public Kaggle datasets, removes exact duplicate files, uses a seeded image-level train/validation split, fine-tunes EfficientNetV2S, and reports class-wise metrics.

> **Important:** This is an educational research prototype, not a medical device. It is not validated for diagnosis, triage, or clinical use. Do not use its predictions to make healthcare decisions.

The notebook targets Google Colab with a GPU and Google Drive. It downloads multiple external datasets (multi-GB total); dataset availability, versions, and licenses are controlled by their owners. Model weights and datasets are not included in this repository.


## 1. Environment and data acquisition

Install the project dependencies, then mount Google Drive for generated model artifacts.

In [ ]:
# Core dependencies. This works when the notebook is opened by itself in Colab.
%pip install -q "tensorflow>=2.15,<2.20" "numpy>=1.26,<2.1" "Pillow>=10,<13" "matplotlib>=3.8,<4" "scikit-learn>=1.4,<2" "kagglehub>=0.3,<1" "lime>=0.2.0.1" "scikit-image>=0.22,<1"


In [ ]:
# This project is designed for Google Colab; Drive is used for model artifacts.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/chest_xray_project')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import kagglehub
path_main  = kagglehub.dataset_download('tawsifurrahman/tuberculosis-tb-chest-xray-dataset')
print('TB/Normal dataset:', path_main)
path_tb2   = kagglehub.dataset_download('raddar/tuberculosis-chest-xrays-shenzhen')
print('TB dataset #2 (Shenzhen):', path_tb2)
path_tb3   = kagglehub.dataset_download('raddar/tuberculosis-chest-xrays-montgomery')
print('TB Montgomery:', path_tb3)
path_tb4   = kagglehub.dataset_download('kmader/pulmonary-chest-xray-abnormalities')
print('TB Shenzhen+Montgomery (kmader):', path_tb4)
path_tb5   = kagglehub.dataset_download('vbookshelf/tbx11k-simplified')
print('TBX11K:', path_tb5)
path_tb6   = kagglehub.dataset_download('vbookshelf/da-and-db-tb-chest-x-ray-datasets')
print('DA & DB:', path_tb6)
path_tb7   = kagglehub.dataset_download('raddar/drug-resistant-tuberculosis-xrays')
print('Drug-resistant TB (Belarus):', path_tb7)
path_tb8   = kagglehub.dataset_download('raddar/chest-xrays-tuberculosis-from-india')
print('TB from India:', path_tb8)
path_pn    = kagglehub.dataset_download('yusufmurtaza01/chest-xray-pneumonia-balanced-dataset')
print('Pneumonia dataset:', path_pn)
path_nr    = kagglehub.dataset_download('alifrahman/chestxraydataset')
print('Normal dataset:', path_nr)
path_other = kagglehub.dataset_download('ibombonato/xray-body-images-in-png-unifesp-competion')
print('Other X-ray dataset:', path_other)
path_other_generic = kagglehub.dataset_download('puneet6060/intel-image-classification')
print('Other generic-image dataset:', path_other_generic)

In [ ]:
import os, shutil, random, json, hashlib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
DATA_MAIN    = Path(path_main)
DATA_TB2     = Path(path_tb2)
DATA_TB3     = Path(path_tb3)
DATA_TB4     = Path(path_tb4)
DATA_TB5     = Path(path_tb5)
DATA_TB6     = Path(path_tb6)
DATA_TB7     = Path(path_tb7)
DATA_TB8     = Path(path_tb8)
DATA_PN      = Path(path_pn)
DATA_NR      = Path(path_nr)
DATA_OTHER   = Path(path_other)
DATA_OTHER_GENERIC = Path(path_other_generic)
NEW_BASE         = '/content/dataset_split'
DRIVE_DIR        = '/content/drive/MyDrive/chest_xray_project'
SAVE_PATH        = f'{DRIVE_DIR}/best_model.keras'
SAVEDMODEL_PATH  = f'{DRIVE_DIR}/best_model_savedmodel'
CONFIG_PATH      = f'{DRIVE_DIR}/model_config.json'
IMG_SIZE         = (180, 180)
BATCH_SIZE       = 32
VAL_RATIO        = 0.15
SEED             = 42
USE_SEG          = False  # Optional; requires requirements-lung-segmentation.txt
MAX_PER_CLASS    = 5000
AUTOTUNE         = tf.data.AUTOTUNE
CLASS_NAMES      = ['normal', 'other', 'pneumonia', 'tuberculosis']
NUM_CLASSES      = len(CLASS_NAMES)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Config ready')

## 2. Data preparation

Image files are collected from the sources below, checked for decode errors, deduplicated by exact file hash, and copied into class folders. The split is **image-level**, not patient-level or source-level; this is a major evaluation limitation.

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def is_valid_image(path: Path) -> bool:
    """Return True only if PIL can fully decode the image."""
    if path.suffix.lower() not in VALID_EXTS:
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            img.load()
        return True
    except Exception:
        return False
def file_hash(path: Path, block_size=65536) -> str:
    h = hashlib.md5()
    try:
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(block_size), b''):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return ''
def collect_images_fast(root: Path, folder_keywords: list) -> list:
    results = []
    for item in root.rglob('*'):
        if not item.is_dir():
            continue
        if any(kw in item.name.lower() for kw in folder_keywords):
            imgs = [f for f in item.iterdir()
                    if f.is_file() and f.suffix.lower() in VALID_EXTS]
            print(f'   {item.parent.name}/{item.name}: {len(imgs)}')
            results.extend(imgs)
    return results
print('Image helpers defined')

# ── Lung Segmentation ─────────────────────────────────────────────────────────
import numpy as _np_seg
from PIL import Image as _PIL_seg

_inferer = None

def _get_inferer():
    global _inferer
    if _inferer is None:
        try:
            from lungmask import LMInferer
        except ImportError as exc:
            raise ImportError(
                'Lung segmentation is optional. Install requirements-lung-segmentation.txt '
                'and set USE_SEG=True to enable it.'
            ) from exc
        _inferer = LMInferer(modelname='R231')
    return _inferer

def _get_bounding_box(mask_np, pad_ratio=0.02):
    """Return (r_min, r_max, c_min, c_max) of the non-zero region with padding."""
    rows = _np_seg.any(mask_np, axis=1)
    cols = _np_seg.any(mask_np, axis=0)
    if not rows.any() or not cols.any():
        return None
    r_min, r_max = _np_seg.where(rows)[0][[0, -1]]
    c_min, c_max = _np_seg.where(cols)[0][[0, -1]]
    h, w = mask_np.shape
    pad_r = max(1, int((r_max - r_min) * pad_ratio))
    pad_c = max(1, int((c_max - c_min) * pad_ratio))
    return (
        max(0, r_min - pad_r),
        min(h, r_max + pad_r),
        max(0, c_min - pad_c),
        min(w, c_max + pad_c),
    )

def apply_lung_segmentation(img_pil, min_lung_ratio=0.05):
    """
    Use lungmask R231 model to segment the lung region from a PIL image,
    then crop the image to the bounding box of the segmented lung.

    lungmask expects a SimpleITK image (or numpy array with shape HxW or HxWxC).
    It returns a label map where 0=background, 1=left lung, 2=right lung.

    Falls back to the original image if:
    - The mask is too small (< min_lung_ratio of total pixels)
    - Any exception occurs
    """
    try:
        import SimpleITK as sitk

        # lungmask works on grayscale; convert PIL → numpy uint8 HxW
        gray = _np_seg.array(img_pil.convert('L'), dtype=_np_seg.uint8)

        # wrap in SimpleITK
        sitk_img = sitk.GetImageFromArray(gray)

        # run inference  (returns HxW label map)
        seg = _get_inferer().apply(sitk_img)          # numpy HxW, values 0/1/2

        # combine left + right lung into one binary mask
        mask = (seg > 0).astype(_np_seg.uint8)

        h, w = mask.shape
        if mask.sum() < min_lung_ratio * h * w:
            return img_pil                            # fallback: mask too small

        bbox = _get_bounding_box(mask)
        if bbox is None:
            return img_pil                            # fallback: empty mask

        r_min, r_max, c_min, c_max = bbox
        return img_pil.crop((c_min, r_min, c_max, r_max))

    except ImportError:
        raise
    except Exception:
        return img_pil                                # fallback: segmentation failure


In [ ]:
def collect_shenzhen_tb(root: Path) -> list:
    for candidate in [root / 'images' / 'images', root / 'images', root]:
        if candidate.exists():
            imgs = [f for f in candidate.iterdir()
                    if f.is_file() and f.suffix.lower() in VALID_EXTS
                    and f.stem.split('_')[-1] == '1']
            if imgs:
                print(f'  Shenzhen TB: {len(imgs)}')
                return imgs
    print('  Shenzhen: folder not found')
    return []

def collect_montgomery_tb(root: Path) -> list:
    for candidate in [root / 'MontgomerySet' / 'CXR_png',
                      root / 'Montgomery' / 'CXR_png']:
        if candidate.exists():
            imgs = [f for f in candidate.iterdir()
                    if f.is_file() and f.suffix.lower() in VALID_EXTS
                    and f.stem.split('_')[-1] == '1']
            print(f'  Montgomery TB: {len(imgs)}')
            return imgs
    imgs = [f for f in root.rglob('*')
            if f.is_file() and f.suffix.lower() in VALID_EXTS
            and f.stem.split('_')[-1] == '1']
    print(f'  Montgomery TB (fallback): {len(imgs)}')
    return imgs

def collect_kmader_tb(root: Path) -> list:
    imgs = [f for f in root.rglob('*')
            if f.is_file() and f.suffix.lower() in VALID_EXTS
            and f.stem.split('_')[-1] == '1']
    print(f'  kmader TB: {len(imgs)}')
    return imgs

def collect_tbx11k_tb(root: Path) -> list:
    imgs = []
    for folder in ['active_tb', 'latent_tb']:
        for split in ['train', 'val']:
            d = root / 'imgs' / split / folder
            if d.exists():
                imgs.extend([f for f in d.iterdir()
                              if f.is_file() and f.suffix.lower() in VALID_EXTS])
    if not imgs:
        imgs = [f for f in root.rglob('*')
                if f.is_file() and f.suffix.lower() in VALID_EXTS
                and ('active_tb' in str(f) or 'latent_tb' in str(f))]
    print(f'  TBX11K TB: {len(imgs)}')
    return imgs

def collect_da_db_tb(root: Path) -> list:
    imgs = [f for f in root.rglob('*')
            if f.is_file() and f.suffix.lower() in VALID_EXTS
            and f.stem.lower().startswith('p')]
    print(f'  DA+DB TB: {len(imgs)}')
    return imgs

def collect_belarus_tb(root: Path) -> list:
    imgs = [f for f in root.rglob('*')
            if f.is_file() and f.suffix.lower() in VALID_EXTS]
    print(f'  Belarus drug-resistant TB: {len(imgs)}')
    return imgs

def collect_india_tb(root: Path) -> list:
    tb_keywords = ['tb', 'tuberculosis', 'positive']
    imgs = [f for f in root.rglob('*')
            if f.is_file() and f.suffix.lower() in VALID_EXTS
            and any(kw in str(f).lower() for kw in tb_keywords)]
    print(f'  India TB: {len(imgs)}')
    return imgs

folder_map = {cls: [] for cls in CLASS_NAMES}

print('Collecting tuberculosis...')
folder_map['tuberculosis'] = collect_images_fast(
    DATA_MAIN, ['tuberculosis', 'tb', 'turberculosis']
)
folder_map['tuberculosis'] += collect_shenzhen_tb(DATA_TB2)
folder_map['tuberculosis'] += collect_montgomery_tb(DATA_TB3)
folder_map['tuberculosis'] += collect_kmader_tb(DATA_TB4)
folder_map['tuberculosis'] += collect_tbx11k_tb(DATA_TB5)
folder_map['tuberculosis'] += collect_da_db_tb(DATA_TB6)
folder_map['tuberculosis'] += collect_belarus_tb(DATA_TB7)
folder_map['tuberculosis'] += collect_india_tb(DATA_TB8)
print(f'  tuberculosis total (before dedup): {len(folder_map["tuberculosis"])}')

print('Collecting normal...')
normal_nr   = collect_images_fast(DATA_NR,   ['normal'])
normal_main = collect_images_fast(DATA_MAIN, ['normal'])
folder_map['normal'] = normal_nr + normal_main

print('Collecting pneumonia...')
folder_map['pneumonia'] = collect_images_fast(DATA_PN, ['pneumonia'])

print('Collecting other...')
other_imgs = []
for folder in ['train', 'test']:
    p = DATA_OTHER / 'images' / folder
    if p.exists():
        imgs = [f for f in p.iterdir()
                if f.is_file() and f.suffix.lower() in VALID_EXTS]
        print(f'  {folder}: {len(imgs)}')
        other_imgs.extend(imgs)
generic_imgs = [f for f in DATA_OTHER_GENERIC.rglob('*')
                if f.is_file() and f.suffix.lower() in VALID_EXTS]
random.shuffle(generic_imgs)
n_generic_cap = max(1, len(other_imgs) // 2)
generic_imgs  = generic_imgs[:n_generic_cap]
print(f'  generic (capped): {len(generic_imgs)}')
folder_map['other'] = other_imgs + generic_imgs

for cls in CLASS_NAMES:
    print(f'  {cls:15s}: {len(folder_map[cls]):>5} images')
print(f'  {"TOTAL":15s}: {sum(len(v) for v in folder_map.values()):>5} images')

In [ ]:
print(f'Lung segmentation enabled: {USE_SEG} (lungmask R231)')
if USE_SEG:
    try:
        import lungmask  # noqa: F401
        import SimpleITK  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            'Install requirements-lung-segmentation.txt before enabling USE_SEG.'
        ) from exc

if os.path.exists(NEW_BASE):
    shutil.rmtree(NEW_BASE)
for split in ['train', 'val']:
    for cls in CLASS_NAMES:
        os.makedirs(os.path.join(NEW_BASE, split, cls), exist_ok=True)
seen_hashes = set()
class_stats = {}
for cls in CLASS_NAMES:
    files = folder_map[cls].copy()
    random.shuffle(files)
    clean_files = []
    n_corrupt, n_dup = 0, 0
    for f in files:
        if len(clean_files) >= MAX_PER_CLASS:
            break
        if not is_valid_image(f):
            n_corrupt += 1
            continue
        h = file_hash(f)
        if h and h in seen_hashes:
            n_dup += 1
            continue
        if h:
            seen_hashes.add(h)
        clean_files.append(f)
    n_val   = max(1, int(len(clean_files) * VAL_RATIO))
    val_f   = clean_files[:n_val]
    train_f = clean_files[n_val:]
    used_names = set()
    for split_name, split_files in [('train', train_f), ('val', val_f)]:
        for f in split_files:
            name = f.name
            if name in used_names:
                name = f'{f.parent.name}_{name}'
            used_names.add(name)
            dst = os.path.join(NEW_BASE, split_name, cls, name)
            if USE_SEG:
                try:
                    img_pil  = Image.open(str(f)).convert('RGB')
                    img_pil  = apply_lung_segmentation(img_pil)
                    dst = os.path.splitext(dst)[0] + '.png'
                    img_pil.save(dst)
                except ImportError:
                    raise
                except Exception:
                    shutil.copy(str(f), dst)
            else:
                shutil.copy(str(f), dst)
    class_stats[cls] = {'train': len(train_f), 'val': len(val_f)}
    print(f'  {cls:15s}: {len(train_f):>4} train / {len(val_f):>3} val  '
          f'(skipped: {n_corrupt} corrupt, {n_dup} duplicate)')
total_tr = sum(v['train'] for v in class_stats.values())
total_vl = sum(v['val']   for v in class_stats.values())
print(f'\n  TOTAL          : {total_tr:>4} train / {total_vl:>3} val')
if total_tr == 0 or any(v['train'] == 0 for v in class_stats.values()):
    raise RuntimeError('train.')

In [ ]:
def _random_gaussian_noise(image, stddev_max=8.0):
    stddev = tf.random.uniform([], 0.0, stddev_max)
    noise = tf.random.normal(tf.shape(image), mean=0.0, stddev=stddev)
    return image + noise
def _random_jpeg_artifact(image):
    def _apply(img):
        quality = int(np.random.randint(30, 90))
        img_uint8 = tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8)
        encoded = tf.io.encode_jpeg(img_uint8, quality=quality)
        decoded = tf.io.decode_jpeg(encoded, channels=3)
        return tf.cast(decoded, tf.float32)
    result = tf.py_function(func=_apply, inp=[image], Tout=tf.float32)
    result.set_shape(image.shape)
    return result
def _random_slight_blur(image, max_sigma=1.0):
    sigma = tf.random.uniform([], 0.0, max_sigma)
    kernel_size = 5
    x = tf.range(-(kernel_size // 2), kernel_size // 2 + 1, dtype=tf.float32)
    g = tf.exp(-(x ** 2) / (2.0 * tf.maximum(sigma, 1e-3) ** 2))
    g = g / tf.reduce_sum(g)
    kernel_1d = g[:, tf.newaxis]
    kernel_2d = kernel_1d @ tf.transpose(kernel_1d)
    kernel = tf.reshape(kernel_2d, [kernel_size, kernel_size, 1, 1])
    kernel = tf.tile(kernel, [1, 1, 3, 1])
    img = image[tf.newaxis, ...]
    blurred = tf.nn.depthwise_conv2d(img, kernel, strides=[1, 1, 1, 1], padding='SAME')
    return blurred[0]
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.15)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    image = tf.cond(
        tf.random.uniform([]) < 0.3,
        lambda: _random_gaussian_noise(image),
        lambda: image
    )
    image = tf.cond(
        tf.random.uniform([]) < 0.3,
        lambda: _random_jpeg_artifact(image),
        lambda: image
    )
    image = tf.cond(
        tf.random.uniform([]) < 0.2,
        lambda: _random_slight_blur(image),
        lambda: image
    )
    image = tf.clip_by_value(image, 0, 255)
    return image, label
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(NEW_BASE, 'train'),
    image_size=IMG_SIZE,
    batch_size=None,
    label_mode='categorical',
    shuffle=True,
    seed=SEED,
    class_names=CLASS_NAMES
)
val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(NEW_BASE, 'val'),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False,
    class_names=CLASS_NAMES
)
assert train_ds_raw.class_names == CLASS_NAMES, (
    f'    train  : {train_ds_raw.class_names} != {CLASS_NAMES}'
)
assert val_ds_raw.class_names == CLASS_NAMES, (
    f'    val  : {val_ds_raw.class_names} != {CLASS_NAMES}'
)
print(' Class order verified:', CLASS_NAMES)
train_ds = (
    train_ds_raw
    .shuffle(300, seed=SEED, reshuffle_each_iteration=True)
    .map(augment, num_parallel_calls=2)
    .batch(BATCH_SIZE)
    .prefetch(2)
)
val_ds = val_ds_raw.prefetch(2)
print('Pipeline ready (RAM-optimized: image-level shuffle buffer, capped prefetch)')

In [ ]:
train_counts  = np.array([class_stats[c]['train'] for c in CLASS_NAMES])
weights       = train_counts.sum() / (NUM_CLASSES * train_counts)
class_weights = dict(enumerate(weights))
for i, cls in enumerate(CLASS_NAMES):
    print(f'  {cls:15s}: {train_counts[i]:>4} imgs  weight={weights[i]:.3f}')

## 3. Model training

EfficientNetV2S is initialized with ImageNet weights. Training runs in two phases: a frozen-backbone classifier head, followed by partial fine-tuning.

In [ ]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Precision policy:', tf.keras.mixed_precision.global_policy().name)
base_model = tf.keras.applications.EfficientNetV2S(
    include_top=False,
    weights='imagenet',
    input_shape=(180, 180, 3),
    include_preprocessing=True
)
base_model.trainable = False
print(f'Base layers: {len(base_model.layers)}')
inputs  = tf.keras.Input(shape=(180, 180, 3))
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.BatchNormalization()(x)
x       = tf.keras.layers.Dropout(0.4)(x)
x       = tf.keras.layers.Dense(512, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x       = tf.keras.layers.BatchNormalization()(x)
x       = tf.keras.layers.Dropout(0.3)(x)
x       = tf.keras.layers.Dense(256, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x       = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
model = tf.keras.Model(inputs, outputs)
model.summary(line_length=80)

In [ ]:
import gc
gc.collect()
print(' Memory collected')
SAVE_PATH_LOCAL = '/content/best_model.weights.h5'
callbacks_phase1 = [
    tf.keras.callbacks.ModelCheckpoint(
        SAVE_PATH_LOCAL,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=2, min_lr=1e-7, verbose=1
    )
]
print(' Callbacks ready')

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
print(' Phase 1: Training head (base frozen)...')
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks_phase1,
    class_weight=class_weights
)
model.save(SAVE_PATH)
print(f' Phase 1 model saved to Drive: {model.input_shape}')

In [ ]:
print(' Phase 2: Fine-tuning (top 80 layers)...')
base_model.trainable = True
for layer in base_model.layers[:-80]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
print(f'Trainable base layers: {sum(1 for l in base_model.layers if l.trainable)}')
import gc
gc.collect()
FT_BATCH_SIZE = 16
train_ds_ft = (
    train_ds_raw
    .shuffle(300, seed=SEED, reshuffle_each_iteration=True)
    .map(augment, num_parallel_calls=2)
    .batch(FT_BATCH_SIZE)
    .prefetch(2)
)
val_ds_ft = val_ds_raw.unbatch().batch(FT_BATCH_SIZE).prefetch(2)
SAVE_PATH_LOCAL_FT = '/content/best_model_ft.weights.h5'
ft_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        SAVE_PATH_LOCAL_FT,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=3, min_lr=1e-8, verbose=1
    )
]
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
history_ft = model.fit(
    train_ds_ft,
    validation_data=val_ds_ft,
    epochs=5,
    callbacks=ft_callbacks,
    class_weight=class_weights
)
model.load_weights(SAVE_PATH_LOCAL_FT)
model.save(SAVE_PATH)
print(f' Final model saved: {SAVE_PATH}')
print(f'   Input shape: {model.input_shape}')
import time
print('\n  Warm-up Inference...')
dummy_input = np.zeros((1, *IMG_SIZE, 3), dtype=np.float32)
t_cold = time.perf_counter()
_ = model.predict(dummy_input, verbose=0)
cold_time_s = time.perf_counter() - t_cold
warm_times = []
for _ in range(3):
    t0 = time.perf_counter()
    _ = model.predict(dummy_input, verbose=0)
    warm_times.append(time.perf_counter() - t0)

## 4. Evaluation and export

The reported metrics below are from the notebook's previously saved validation run; outputs have been cleared so readers can reproduce them. They are **not** an independent test-set or external-site evaluation.

In [ ]:
model_config = {
    'class_names': CLASS_NAMES,
    'img_size': list(IMG_SIZE),
    'num_classes': NUM_CLASSES,
    'requires_warmup': True,
    'expected_warm_inference_ms': float(np.mean(warm_times) * 1000),
    'expected_cold_first_call_s': float(cold_time_s),
}
os.makedirs(DRIVE_DIR, exist_ok=True)
with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    json.dump(model_config, f, ensure_ascii=False, indent=2)
print(f' Config saved: {CONFIG_PATH}')
print(json.dumps(model_config, ensure_ascii=False, indent=2))
model.export(SAVEDMODEL_PATH)
def dir_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            total += os.path.getsize(os.path.join(root, fn))
    return total / (1024 * 1024)
print(f' Keras model size on disk : {os.path.getsize(SAVE_PATH) / (1024*1024):.2f} MB')
print(f' SavedModel size on disk  : {dir_size_mb(SAVEDMODEL_PATH):.2f} MB')

In [ ]:
all_acc      = history.history['accuracy']     + history_ft.history['accuracy']
all_val_acc  = history.history['val_accuracy'] + history_ft.history['val_accuracy']
all_loss     = history.history['loss']         + history_ft.history['loss']
all_val_loss = history.history['val_loss']     + history_ft.history['val_loss']
split_ep     = len(history.history['accuracy'])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, tr, vl, title in [
    (ax1, all_acc,  all_val_acc,  'Accuracy'),
    (ax2, all_loss, all_val_loss, 'Loss')
]:
    ax.plot(tr, label='Train')
    ax.plot(vl, label='Validation')
    ax.axvline(split_ep - 0.5, color='red', linestyle='--', label='Fine-tune start')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('Chest X-Ray Classifier  Training History')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, auc as sklearn_auc,
    precision_recall_fscore_support
)

loss_val, acc_val, auc_val = model.evaluate(val_ds, verbose=0)

y_true, y_pred_idx, y_prob = [], [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_idx.extend(np.argmax(preds, axis=1))
    y_prob.extend(preds)

y_true     = np.array(y_true)
y_pred_idx = np.array(y_pred_idx)
y_prob     = np.array(y_prob)

print(f"Loss     : {loss_val:.4f}")
print(f"Accuracy : {acc_val:.4f}  ({acc_val*100:.2f}%)")
print(f"AUC      : {auc_val:.4f}")
print(classification_report(y_true, y_pred_idx, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred_idx)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False)
plt.title("Confusion Matrix — Validation Set")
plt.tight_layout()
plt.show()

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred_idx, labels=range(NUM_CLASSES), zero_division=0
)

print(f"{'Class':<16} {'Precision':>10} {'Sensitivity':>12} {'F1':>8} {'Specificity':>12}")

for i, cls in enumerate(CLASS_NAMES):
    # Specificity = TN / (TN + FP)  computed per class (one-vs-rest)
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    print(f"{cls:<16} {precision[i]:>10.4f} {recall[i]:>12.4f} {f1[i]:>8.4f} {spec:>12.4f}")


y_true_bin = np.zeros((len(y_true), NUM_CLASSES))
y_true_bin[np.arange(len(y_true)), y_true] = 1

fig, ax = plt.subplots(figsize=(8, 6))
for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc_i   = sklearn_auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{cls}  (AUC = {roc_auc_i:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("1 - Specificity  (FPR)")
ax.set_ylabel("Sensitivity  (TPR)")
ax.set_title("ROC Curves — One-vs-Rest per Class")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

macro_auc = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")
print(f"Macro AUC (OvR): {macro_auc:.4f}")

## 5. Optional explanation and single-image demo

The LIME cell requires `lime` and `scikit-image`. The final upload/prediction cell is interactive and intended for Colab. Image-format screening and the `other` class are heuristic/model behavior only; neither validates that an image is a chest X-ray.

In [ ]:
import lime
import lime.lime_image
from skimage.segmentation import mark_boundaries

def lime_explain(image_path, model, class_names, img_size, num_samples=500):
    from PIL import Image as PILImage
    import numpy as np

    img_raw = PILImage.open(image_path).convert("RGB").resize((img_size[1], img_size[0]))
    img_np  = np.array(img_raw, dtype=np.float32)

    def batch_predict(images):
        return model.predict(np.array(images, dtype=np.float32), verbose=0)

    explainer    = lime.lime_image.LimeImageExplainer(random_state=42)
    explanation  = explainer.explain_instance(
        img_np,
        batch_predict,
        top_labels=len(class_names),
        hide_color=0,
        num_samples=num_samples,
    )

    fig, axes = plt.subplots(1, len(class_names), figsize=(4 * len(class_names), 4))
    for idx, cls in enumerate(class_names):
        label_idx = idx
        temp, mask = explanation.get_image_and_mask(
            label_idx, positive_only=False, num_features=10, hide_rest=False
        )
        axes[idx].imshow(mark_boundaries(temp.astype(np.uint8), mask))
        axes[idx].set_title(cls)
        axes[idx].axis("off")
    plt.suptitle("LIME Explanations per Class")
    plt.tight_layout()
    plt.show()

    return explanation


from google.colab import files
uploaded_lime = files.upload()
lime_img_path = list(uploaded_lime.keys())[0]
lime_explanation = lime_explain(lime_img_path, model, CLASS_NAMES, IMG_SIZE)

## 6. Reload a saved model

Use this section when starting a new runtime after training. If running the notebook from top to bottom, the model is already available.

In [ ]:
import json
from pathlib import Path
import tensorflow as tf
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/chest_xray_project')
SAVE_PATH = DRIVE_DIR / 'best_model.keras'
CONFIG_PATH = DRIVE_DIR / 'model_config.json'

if not SAVE_PATH.exists():
    raise FileNotFoundError(f'Model not found: {SAVE_PATH}. Train or upload the model first.')
model = tf.keras.models.load_model(SAVE_PATH)
if CONFIG_PATH.exists():
    model_config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    CLASS_NAMES = model_config['class_names']
    IMG_SIZE = tuple(model_config['img_size'])
    NUM_CLASSES = model_config['num_classes']
    print(f'Loaded model configuration: {CONFIG_PATH}')
else:
    CLASS_NAMES = ['normal', 'other', 'pneumonia', 'tuberculosis']
    NUM_CLASSES = len(CLASS_NAMES)
    IMG_SIZE = tuple(model.input_shape[1:3])
    print('Configuration file not found; using notebook defaults.')

model_img_size = tuple(model.input_shape[1:3])
if IMG_SIZE != model_img_size:
    print(f'Configured image size {IMG_SIZE} differs from model input {model_img_size}; using model input size.')
    IMG_SIZE = model_img_size
print('Model ready | image size:', IMG_SIZE, '| classes:', CLASS_NAMES)


In [ ]:
from google.colab import files
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def is_xray_like(img_raw: Image.Image) -> tuple[bool, dict]:
    """Heuristic image-format screen; it is not a medical image validator."""
    img_np = np.array(img_raw.convert('RGB'), dtype=np.float32)
    r, g, b = img_np[:, :, 0], img_np[:, :, 1], img_np[:, :, 2]
    color_diff = (np.abs(r - g).mean() + np.abs(g - b).mean() + np.abs(r - b).mean()) / 3
    max_rgb, min_rgb = np.max(img_np, axis=2), np.min(img_np, axis=2)
    saturation = np.where(max_rgb == 0, 0, (max_rgb - min_rgb) / (max_rgb + 1e-7))
    gray = np.array(img_raw.convert('L'), dtype=np.float32)
    gray_rgb = np.stack([gray, gray, gray], axis=-1)
    metrics = {
        'color_diff': float(color_diff),
        'saturation': float(saturation.mean()),
        'gray_dist': float(np.mean(np.abs(img_np - gray_rgb))),
        'color_variance': float(np.std([r.mean(), g.mean(), b.mean()])),
        'brightness': float(gray.mean()),
    }
    is_colorful = (
        metrics['color_diff'] > 18 or metrics['saturation'] > 0.08
        or metrics['gray_dist'] > 12 or metrics['color_variance'] > 8
        or metrics['brightness'] > 235
    )
    return is_colorful, metrics

def predict_image(img_raw: Image.Image, model, class_names, img_size) -> dict:
    """Run the model on one image. This is a research demo, not clinical software."""
    if img_raw is None or not isinstance(img_raw, Image.Image):
        raise ValueError('Expected a PIL image.')
    img_rgb = img_raw.convert('RGB')
    if img_rgb.width < 10 or img_rgb.height < 10:
        raise ValueError(f'Image is too small: {img_rgb.size}')
    is_colorful, color_metrics = is_xray_like(img_rgb)
    if is_colorful:
        return {'rejected': True, 'reason': 'not_xray', 'metrics': color_metrics, 'image': img_rgb}
    img_resized = img_rgb.resize((img_size[1], img_size[0]))
    img_array = np.asarray(img_resized, dtype=np.float32)[np.newaxis, ...]
    probs = model.predict(img_array, verbose=0)[0]
    pred_class = class_names[int(np.argmax(probs))]
    is_other = pred_class == 'other'
    return {
        'rejected': is_other,
        'reason': 'classified_as_other' if is_other else None,
        'pred_class': pred_class,
        'confidence': float(probs.max()),
        'probabilities': dict(zip(class_names, probs.tolist())),
        'image': img_rgb,
    }

def show_prediction(result: dict, class_names):
    img_raw = result['image']
    if result['reason'] == 'not_xray':
        plt.figure(figsize=(5, 4))
        plt.imshow(img_raw)
        plt.axis('off')
        plt.title('Rejected by the color heuristic (not a medical validation)', color='red')
        plt.tight_layout()
        plt.show()
        print('Image-format heuristic metrics:', result['metrics'])
        return
    probs = np.array([result['probabilities'][c] for c in class_names])
    is_other = result['rejected']
    color = 'red' if is_other else 'green'
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(img_raw)
    ax1.axis('off')
    for spine in ax1.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(4)
        spine.set_visible(True)
    ax1.set_title('Predicted class: other' if is_other else 'Model prediction', color=color)
    bars = ax2.barh(class_names, probs * 100,
                    color=['green' if p == probs.max() else 'steelblue' for p in probs])
    ax2.set_xlabel('Model score (%)')
    ax2.set_xlim(0, 100)
    for bar, prob in zip(bars, probs):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                 f'{prob * 100:.1f}%', va='center')
    ax2.set_title('Uncalibrated class scores')
    plt.tight_layout()
    plt.show()
    print(f"Predicted class: {result['pred_class']} ({result['confidence'] * 100:.1f}% score)")

# Optional interactive demo: upload one image in Colab.
uploaded = files.upload()
if not uploaded:
    raise ValueError('No image was uploaded.')
img_path = next(iter(uploaded))
with Image.open(img_path) as image_file:
    image_file.verify()
with Image.open(img_path) as image_file:
    result = predict_image(image_file, model, CLASS_NAMES, IMG_SIZE)
show_prediction(result, CLASS_NAMES)


## Previously recorded validation results

These values are retained from the source notebook's saved run and have not been regenerated during cleanup. The validation set contained 2,144 images.

| Class | Precision | Recall / sensitivity | F1 | Specificity | Support |
|---|---:|---:|---:|---:|---:|
| Normal | 0.9620 | 0.9453 | 0.9536 | 0.9799 | 750 |
| Other | 0.9927 | 0.9838 | 0.9882 | 0.9975 | 555 |
| Pneumonia | 0.9564 | 0.9669 | 0.9616 | 0.9814 | 635 |
| Tuberculosis | 0.9256 | 0.9755 | 0.9499 | 0.9918 | 204 |

**Overall validation accuracy:** 0.9646 · **Macro one-vs-rest AUC:** 0.9979

Because this is a random image-level split over several heterogeneous datasets, duplicates or correlated images across train and validation may remain, and acquisition/source differences may dominate learned signals. These results should be treated as preliminary and may change when upstream datasets or runtime versions change.
